# Delta Lake com Apache Spark (PySpark)

## Cenário: Plataforma de E-commerce

Neste notebook, demonstramos o uso do **Delta Lake** com **Apache Spark (PySpark)** aplicado a um sistema de pedidos de e-commerce.

O cenário modela três entidades:
- **Cliente** — dados cadastrais do comprador
- **Produto** — catálogo com categoria e preço
- **Pedido** — relaciona cliente e produto com status, quantidade e data

Demonstraremos as operações **DDL**, **INSERT**, **UPDATE**, **DELETE** e **Time Travel**.

## Modelo Entidade-Relacionamento

```
┌──────────────┐         ┌──────────────┐
│   CUSTOMER   │         │   PRODUCT    │
├──────────────┤         ├──────────────┤
│ customer_id  │PK       │ product_id   │PK
│ name         │         │ name         │
│ email        │         │ category     │
│ city         │         │ price        │
└──────┬───────┘         └──────┬───────┘
       │                        │
       └──────────┬─────────────┘
                  │
                  ▼
        ┌─────────────────────┐
        │        ORDER        │
        ├─────────────────────┤
        │ order_id     (PK)   │
        │ customer_id  (FK)   │
        │ product_id   (FK)   │
        │ quantity            │
        │ unit_price          │
        │ status              │
        │ order_date          │
        └─────────────────────┘
```

| Entidade | Atributos |
|---|---|
| CUSTOMER | customer_id (PK), name, email, city |
| PRODUCT  | product_id (PK), name, category, price |
| ORDER    | order_id (PK), customer_id (FK), product_id (FK), quantity, unit_price, status, order_date |

## 1. Configuração — SparkSession com Delta Lake

In [2]:
import sys
import os
# Adiciona o diretório src ao path para permitir importação do pacote local
sys.path.append(os.path.abspath("../src"))

from spark_delta_apache import get_spark_delta_session, setup_logging

# Configurar logging e criar SparkSession via pacote local
setup_logging()
spark = get_spark_delta_session("DeltaLakeEcommerce")
setup_logging(spark.sparkContext)
spark


## 2. DDL — Criação das Tabelas

Criamos as três tabelas com `USING delta`. O Spark armazena os dados em formato **Parquet** e mantém o histórico de transações na pasta `_delta_log/`.

In [3]:
import shutil
import os

# Limpeza física dos caminhos para evitar erro de 'Non-empty location'
for table in ['customer_delta', 'product_delta', 'order_delta', 'customer_iceberg', 'product_iceberg', 'order_iceberg']:
    path = f"/app/spark-warehouse/{table}"
    if os.path.exists(path):
        shutil.rmtree(path)
    # Limpa também o caminho do Iceberg se necessário
    ice_path = f"/app/spark-warehouse/iceberg/local/{table}"
    if os.path.exists(ice_path):
        shutil.rmtree(ice_path)

# 2. DDL — Criação das Tabelas
spark.sql("DROP TABLE IF EXISTS customer_delta")
spark.sql("DROP TABLE IF EXISTS product_delta")
spark.sql("DROP TABLE IF EXISTS order_delta")

spark.sql("""
    CREATE TABLE customer_delta (
        customer_id INT,
        name        STRING,
        email       STRING,
        city        STRING
    ) USING delta
""")

spark.sql("""
    CREATE TABLE product_delta (
        product_id INT,
        name       STRING,
        category   STRING,
        price      DOUBLE
    ) USING delta
""")

spark.sql("""
    CREATE TABLE order_delta (
        order_id    INT,
        customer_id INT,
        product_id  INT,
        quantity    INT,
        unit_price  DOUBLE,
        status      STRING,
        order_date  DATE
    ) USING delta
""")

print("Tabelas criadas com sucesso!")
spark.sql("SHOW TABLES").show()

Tabelas criadas com sucesso!
+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|  default|customer_delta|      false|
|  default|   order_delta|      false|
|  default| product_delta|      false|
+---------+--------------+-----------+



## 3. INSERT — Inserção de Dados

Cada `INSERT` gera uma nova **versão** no `_delta_log`, registrando quais arquivos Parquet foram adicionados.

In [4]:
# Inserir clientes
spark.sql("""
    INSERT INTO customer_delta VALUES
        (1, 'Ana Silva',   'ana@email.com',   'São Paulo'),
        (2, 'Bruno Costa', 'bruno@email.com', 'Rio de Janeiro'),
        (3, 'Carla Lima',  'carla@email.com', 'Curitiba')
""")

# Inserir produtos
spark.sql("""
    INSERT INTO product_delta VALUES
        (1, 'Notebook',      'Eletronicos', 3500.00),
        (2, 'Mouse',         'Eletronicos',   89.90),
        (3, 'Cadeira Gamer', 'Moveis',      1200.00)
""")

# Inserir pedidos
spark.sql("""
    INSERT INTO order_delta VALUES
        (1, 1, 1, 1, 3500.00, 'pendente', DATE '2024-01-10'),
        (2, 2, 2, 2,   89.90, 'aprovado', DATE '2024-01-11'),
        (3, 3, 3, 1, 1200.00, 'pendente', DATE '2024-01-12')
""")

print("=== Clientes ===")
spark.sql("SELECT * FROM customer_delta").show()

print("=== Produtos ===")
spark.sql("SELECT * FROM product_delta").show()

print("=== Pedidos ===")
spark.sql("SELECT * FROM order_delta").show()

=== Clientes ===


+-----------+-----------+---------------+--------------+
|customer_id|       name|          email|          city|
+-----------+-----------+---------------+--------------+
|          2|Bruno Costa|bruno@email.com|Rio de Janeiro|
|          3| Carla Lima|carla@email.com|      Curitiba|
|          1|  Ana Silva|  ana@email.com|     São Paulo|
+-----------+-----------+---------------+--------------+

=== Produtos ===
+----------+-------------+-----------+------+
|product_id|         name|   category| price|
+----------+-------------+-----------+------+
|         3|Cadeira Gamer|     Moveis|1200.0|
|         1|     Notebook|Eletronicos|3500.0|
|         2|        Mouse|Eletronicos|  89.9|
+----------+-------------+-----------+------+

=== Pedidos ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       1|          1|

## 4. UPDATE — Atualização de Dados

O Delta Lake implementa `UPDATE` como **copy-on-write**: novos arquivos Parquet são escritos com os dados atualizados e o `_delta_log` registra a mudança. Os arquivos originais permanecem intactos (disponíveis para Time Travel).

In [5]:
# Atualizar status do pedido 1 para 'aprovado'
spark.sql("""
    UPDATE order_delta
    SET status = 'aprovado'
    WHERE order_id = 1
""")

# Atualizar preço do Notebook com desconto
spark.sql("""
    UPDATE product_delta
    SET price = 3299.00
    WHERE product_id = 1
""")

print("=== Pedidos após UPDATE ===")
spark.sql("SELECT * FROM order_delta").show()

print("=== Produtos após UPDATE ===")
spark.sql("SELECT * FROM product_delta").show()

=== Pedidos após UPDATE ===
+--------+-----------+----------+--------+----------+--------+----------+
|order_id|customer_id|product_id|quantity|unit_price|  status|order_date|
+--------+-----------+----------+--------+----------+--------+----------+
|       1|          1|         1|       1|    3500.0|aprovado|2024-01-10|
|       3|          3|         3|       1|    1200.0|pendente|2024-01-12|
|       2|          2|         2|       2|      89.9|aprovado|2024-01-11|
+--------+-----------+----------+--------+----------+--------+----------+

=== Produtos após UPDATE ===
+----------+-------------+-----------+------+
|product_id|         name|   category| price|
+----------+-------------+-----------+------+
|         3|Cadeira Gamer|     Moveis|1200.0|
|         1|     Notebook|Eletronicos|3299.0|
|         2|        Mouse|Eletronicos|  89.9|
+----------+-------------+-----------+------+



## 5. DELETE — Remoção de Dados

O `DELETE` também é copy-on-write. Os dados removidos ainda existem nos arquivos físicos e podem ser recuperados via **Time Travel** até que `VACUUM` seja executado.

In [ ]:
# Cancelar (remover) o pedido 3
spark.sql("""
    DELETE FROM order_delta
    WHERE order_id = 3
""")

print("=== Pedidos após DELETE ===")
spark.sql("SELECT * FROM order_delta").show()

=== Pedidos após DELETE ===


## 6. MERGE INTO — Upsert de Dados

O `MERGE INTO` permite **upsert** atômico: atualiza registros existentes e insere novos em uma única operação.

- `WHEN MATCHED` — atualiza registros que existem na tabela alvo
- `WHEN NOT MATCHED` — insere registros novos da fonte

Cada `MERGE` gera uma nova versão no `_delta_log`, preservando o histórico completo.

In [ ]:
# Fonte de dados para upsert:
# - order_id 1: já existe → atualiza status para 'entregue'
# - order_id 4: não existe → insere novo pedido
spark.sql("""
    MERGE INTO order_delta AS target
    USING (
        SELECT 1 AS order_id, 1 AS customer_id, 1 AS product_id,
               1 AS quantity, 3500.00 AS unit_price, 'entregue' AS status,
               DATE '2024-01-10' AS order_date
        UNION ALL
        SELECT 4, 1, 2, 3, 89.90, 'aprovado', DATE '2024-01-15'
    ) AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN
        UPDATE SET status = source.status
    WHEN NOT MATCHED THEN
        INSERT *
""")

print("=== Pedidos após MERGE INTO ===")
spark.sql("SELECT * FROM order_delta ORDER BY order_id").show()

## 7. Time Travel e Histórico de Transações

O Delta Lake mantém um **log de transações** em `_delta_log/`. Cada operação (CREATE, INSERT, UPDATE, DELETE, MERGE) cria uma nova versão numerada, permitindo leitura por **número de versão** ou por **timestamp**.

In [ ]:
from delta.tables import DeltaTable

# Visualizar histórico completo de transações
dt_orders = DeltaTable.forName(spark, "order_delta")
print("=== Histórico de Transações (order_delta) ===")
dt_orders.history().select(
    "version", "timestamp", "operation"
).show(truncate=False)

In [ ]:
# Time Travel: ler a versão 0 (estado após o CREATE TABLE, antes de qualquer INSERT)
print("=== Time Travel — Versão 0 (CREATE TABLE) ===")
spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .table("order_delta") \
    .show()

# Time Travel: ler a versão 1 (após o primeiro INSERT)
print("=== Time Travel — Versão 1 (após INSERT) ===")
spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .table("order_delta") \
    .show()

# Time Travel por timestamp — lê o timestamp da versão 1 dinamicamente do histórico
ts_v1 = dt_orders.history().filter("version = 1").select("timestamp").first()["timestamp"]
print(f"=== Time Travel — por Timestamp (versão 1: {ts_v1}) ===")
spark.read.format("delta") \
    .option("timestampAsOf", str(ts_v1)) \
    .table("order_delta") \
    .show()

# Estado atual
print("=== Estado Atual ===")
spark.sql("SELECT * FROM order_delta").show()

In [ ]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")